# Chapitre 3 — Importer, exporter, faire connaissance avec un jeu de données

## Le problème

Les données arrivent rarement sous forme de dictionnaire Python : elles sont dans des fichiers CSV, des classeurs Excel, du JSON, des bases SQL. La première étape de tout travail consiste à les charger **correctement** (bon séparateur, bons types, bonnes valeurs manquantes), puis à comprendre leur contenu avant de calculer quoi que ce soit.

Ce chapitre couvre :

1. la lecture et l'écriture des formats courants (CSV, Excel, JSON) ;
2. les outils d'inspection : `head`, `tail`, `info`, `describe`, `isna`, `nunique` ;
3. une méthode systématique pour découvrir un fichier inconnu.

In [ ]:
import os
import numpy as np
import pandas as pd

os.makedirs("../sorties", exist_ok=True)     # dossier pour les fichiers produits par le cours

## 1. Lire un CSV

Un CSV (*comma-separated values*) est un fichier texte : une ligne par enregistrement, des valeurs séparées par un caractère. La fonction `pd.read_csv` en possède plus de cinquante paramètres ; une dizaine suffit dans la pratique.

In [ ]:
clients = pd.read_csv("../donnees/demo/clients.csv")
clients.head(3)

### Quand le fichier ne suit pas les conventions

Le fichier `commandes_brutes.csv` utilise le point-virgule comme séparateur, comme beaucoup d'exports Excel en Europe.

> **Prédiction.** Que va produire `pd.read_csv` sans préciser le séparateur ? Combien de colonnes ?

In [ ]:
brut = pd.read_csv("../donnees/demo/commandes_brutes.csv")
brut.shape

In [ ]:
brut.head(2)

L'en-tête ne contient aucune virgule : Pandas crée **une seule colonne**, dont le nom est la ligne d'en-tête entière. Les lignes de données, elles, contiennent une virgule, celle des prix (`79,00 €`) : chacune est donc découpée en deux morceaux, et le premier morceau devient l'index. Le résultat est absurde, et pourtant aucune erreur n'est levée.

Le réflexe est de vérifier `shape` et `head()` juste après chaque lecture.

In [ ]:
brut = pd.read_csv("../donnees/demo/commandes_brutes.csv", sep=";")
brut.shape

### Les paramètres utiles de `read_csv`

| Paramètre | Rôle | Exemple |
|---|---|---|
| `sep` | séparateur | `sep=";"`, `sep="\t"`, `sep="\|"` |
| `index_col` | colonne à utiliser comme index | `index_col="client_id"` |
| `usecols` | ne lire que certaines colonnes | `usecols=["Name", "Age"]` |
| `nrows` | ne lire que les n premières lignes | `nrows=1000` |
| `dtype` | imposer des types | `dtype={"code_postal": "str"}` |
| `parse_dates` | convertir des colonnes en dates | `parse_dates=["date_inscription"]` |
| `na_values` | textes à considérer comme manquants | `na_values=["-", "inconnu"]` |
| `decimal` | séparateur décimal | `decimal=","` |
| `encoding` | encodage du fichier | `encoding="latin-1"` |
| `compression` | fichier compressé | `compression="zip"` (déduit de l'extension) |

In [ ]:
clients = pd.read_csv(
    "../donnees/demo/clients.csv",
    index_col="client_id",
    parse_dates=["date_inscription"],
)
clients.dtypes

In [ ]:
clients.head(3)

L'index contient maintenant les identifiants des clients, et la date d'inscription est une vraie date (`datetime64`), sur laquelle on pourra calculer.

### Lire directement depuis une URL

`read_csv` accepte une adresse web. Les exercices du dossier `drills` utilisent cette possibilité ; des copies locales sont fournies dans `donnees/drills/` pour travailler hors ligne. Le fichier Chipotle est séparé par des tabulations, le fichier des utilisateurs par des barres verticales :

In [ ]:
chipo = pd.read_csv("../donnees/drills/chipotle.tsv", sep="\t")
users = pd.read_csv("../donnees/drills/u.user", sep="|", index_col="user_id")
chipo.shape, users.shape

## 2. Écrire un CSV

In [ ]:
clients.to_csv("../sorties/clients_export.csv")
open("../sorties/clients_export.csv", encoding="utf-8").read()[:180]

Par défaut, `to_csv` écrit aussi l'index. Avec un index sans signification (0, 1, 2…), on ajoute `index=False`, sinon la relecture crée une colonne `Unnamed: 0`.

In [ ]:
produits = pd.read_csv("../donnees/demo/produits.csv")
produits.to_csv("../sorties/produits_avec_index.csv")
pd.read_csv("../sorties/produits_avec_index.csv").head(2)

In [ ]:
produits.to_csv("../sorties/produits.csv", index=False)
pd.read_csv("../sorties/produits.csv").head(2)

La colonne `Unnamed: 0` apparaît d'ailleurs dans le classeur Sakila de l'exercice du chapitre 5 : quelqu'un a exporté un DataFrame sans `index=False`.

Pour un fichier volumineux, la compression se déduit de l'extension :

In [ ]:
produits.to_csv("../sorties/produits.csv.zip", index=False)
pd.read_csv("../sorties/produits.csv.zip").shape

## 3. Excel

La lecture et l'écriture des fichiers `.xlsx` passent par la bibliothèque `openpyxl`, installée avec le projet.

In [ ]:
xl = pd.ExcelFile("../donnees/bstorm/sakila.xlsx")
xl.sheet_names

In [ ]:
actor = pd.read_excel("../donnees/bstorm/sakila.xlsx", sheet_name="actor")
actor.head(3)

`sheet_name=None` charge **toutes** les feuilles dans un dictionnaire `{nom_de_feuille: DataFrame}`.

In [ ]:
feuilles = pd.read_excel("../donnees/bstorm/sakila.xlsx", sheet_name=None)
{nom: f.shape for nom, f in feuilles.items()}

Pour écrire plusieurs feuilles dans un même classeur, on utilise un `ExcelWriter` dans un bloc `with`, qui ferme correctement le fichier.

In [ ]:
with pd.ExcelWriter("../sorties/boutique.xlsx") as writer:
    clients.to_excel(writer, sheet_name="clients")
    produits.to_excel(writer, sheet_name="produits", index=False)

pd.read_excel("../sorties/boutique.xlsx", sheet_name="produits", usecols=["produit_id", "prix"]).head(3)

## 4. JSON

Le JSON est le format des API web. Deux formes reviennent souvent :

- une **liste d'enregistrements** : `[{"col": val, ...}, {...}]` ;
- un **dictionnaire de colonnes** : `{"col": {index: val, ...}, ...}`.

Le paramètre `orient` choisit la forme à l'écriture comme à la lecture.

In [ ]:
produits.head(2).to_json(orient="records", force_ascii=False)

In [ ]:
produits.head(2).to_json(orient="columns", force_ascii=False)

In [ ]:
from io import StringIO
texte = '[{"produit_id": "P01", "prix": 89.9}, {"produit_id": "P02", "prix": 24.5}]'
pd.read_json(StringIO(texte))

`read_json` attend un fichier, une URL ou un objet fichier : le texte brut doit être enveloppé dans `StringIO`. Quand le JSON contient des objets imbriqués, `pd.json_normalize` les aplatit en colonnes (chapitre 9 et chapitre 11).

## 5. Faire connaissance avec un jeu de données

Chargeons le jeu de données de l'exercice principal : 120 ans de Jeux olympiques.

In [ ]:
jo = pd.read_csv("../donnees/bstorm/Exercices_Pandas_athlete_events.csv")
jo.shape

### Regarder les données

In [ ]:
jo.head()

In [ ]:
jo.tail(3)

In [ ]:
jo.sample(3, random_state=1)     # trois lignes au hasard : souvent plus représentatif que head

### La structure : `info()`

`info()` résume en une commande : le nombre de lignes, le type de chaque colonne, le nombre de valeurs **non nulles**, la mémoire occupée.

In [ ]:
jo.info()

> **Mini-problème.** À partir de la sortie de `info()` uniquement, relevez trois informations sur ce jeu de données qu'il faudra prendre en compte avant d'en tirer des conclusions.

### Les valeurs manquantes

In [ ]:
jo.isna().sum()

In [ ]:
(jo.isna().mean() * 100).round(1)      # en pourcentage : la moyenne d'une colonne de booléens

Une valeur manquante n'est pas toujours une anomalie. Dans `Medal`, `NaN` signifie « pas de médaille » : c'est une information. Dans `Age`, c'est une donnée inconnue.

### Les statistiques : `describe()`

In [ ]:
jo.describe()

Par défaut, `describe()` ne résume que les colonnes numériques. `include="all"` ajoute les colonnes texte, avec le nombre de valeurs distinctes (`unique`), la plus fréquente (`top`) et sa fréquence (`freq`).

In [ ]:
jo.describe(include="all")

On peut aussi cibler un type de colonnes avec `select_dtypes`.

In [ ]:
jo.select_dtypes(include="number").columns.tolist(), jo.select_dtypes(include="str").columns.tolist()

### Le nombre de valeurs distinctes

In [ ]:
jo.nunique()

> **Pourquoi ?** La colonne `ID` compte 135 571 valeurs distinctes pour 271 116 lignes. Qu'est-ce que cela indique sur ce qu'est une ligne de ce fichier ?

### Une fonction d'inspection réutilisable

La correction Bstorm de l'exercice JO propose de regrouper ces informations dans un seul tableau. C'est une bonne habitude : on écrit cette fonction une fois, on la réutilise sur chaque nouveau fichier.

In [ ]:
def info(df):
    """Résumé d'un DataFrame : type, valeurs manquantes, valeurs distinctes."""
    return pd.DataFrame({
        "type": df.dtypes,
        "non_nuls": df.count(),
        "manquants": df.isna().sum(),
        "pct_manquants": (df.isna().mean() * 100).round(1),
        "distincts": df.nunique(),
    })

info(jo)

### La mémoire

In [ ]:
print(round(jo.memory_usage(deep=True).sum() / 1e6, 1), "Mo en mémoire")

Le fichier fait 41 Mo sur disque mais le DataFrame en occupe davantage. Les colonnes texte et leurs millions de chaînes en sont la cause. Une conversion des colonnes répétitives (`Sex`, `Season`, `NOC`, `Medal`…) en `category` réduit fortement cette taille (chapitre 5).

## Une méthode pour découvrir un fichier

1. **Lire** et vérifier immédiatement `shape` et `head()` (séparateur, en-tête, encodage).
2. **Structure** : `info()` pour les types et les non-nuls.
3. **Manquants** : `isna().sum()`, et se demander ce que signifie chaque `NaN`.
4. **Distributions** : `describe()` et `describe(include="all")`.
5. **Cardinalités** : `nunique()` pour repérer identifiants, catégories, textes libres.
6. **Granularité** : qu'est-ce qu'une ligne ? (un athlète ? une participation ? une épreuve ?)

## Pièges fréquents

| Situation | Symptôme | Réflexe |
|---|---|---|
| Mauvais séparateur | une seule colonne | `sep=` et vérifier `shape` |
| Index exporté par erreur | colonne `Unnamed: 0` | `index=False` à l'écriture, ou `index_col=0` à la lecture |
| Dates lues comme du texte | type `str` | `parse_dates=` ou `pd.to_datetime` |
| Codes postaux ou identifiants à zéros initiaux | `0123` devient `123` | `dtype={"col": "str"}` |
| Décimales avec une virgule | colonne lue en texte | `decimal=","` |
| `describe()` sans les colonnes texte | colonnes absentes du résumé | `include="all"` |

## Exercices

- Exercice Bstorm **JO**, questions **1 à 8** (`exercices/bstorm/01_pandas_JO/`). Les données sont aussi dans `donnees/bstorm/`.
- Renforcement : drill **D01 Occupation**, puis **D01 Chipotle** étapes 1 à 12 (`exercices/drills/`).

## À retenir

- `read_csv`, `read_excel`, `read_json` et leurs symétriques `to_csv`, `to_excel`, `to_json`.
- Toujours vérifier `shape` et `head()` après une lecture : une erreur de lecture est souvent silencieuse.
- `info()`, `isna().sum()`, `describe(include="all")`, `nunique()` : quatre commandes pour comprendre un fichier.
- Une valeur manquante peut être une absence d'information… ou une information.
- Identifier la granularité d'une ligne avant tout calcul.